This notebook shows the logic used to add attributes to the hdf5 file that tell what files are a part of which datasets and class numbers.
Since the lower numbers are subsets of top_50, the filenames can be filtered and matching 
labels can be added. The log-melspectrograms are all already contained.

In [2]:
import os

import h5py
import torch
import numpy as np

import utils.utility_funcs as my_funcs

In [11]:
PATH_TO_VALID_FILENAMES_DIR = r"C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\valid_filenames"
PATH_TO_LABEL_DICT_DIR = r"C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\labels_by_count"
PATH_TO_DATA_HDF5 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\data_cl_complete.hdf5'

In [9]:
fn_files = os.scandir(PATH_TO_VALID_FILENAMES_DIR)

valid_filenames = {}
for fn_file in fn_files:
    split_fname = fn_file.name.split('_')
    dataset, datasplit, nr_of_classes = split_fname[0], split_fname[1], split_fname[3][3:5]
    print(f"dataset: {dataset}, datasplit: {datasplit}, nr_of_classes: {nr_of_classes}")
    key = f"{dataset}_{datasplit}_{nr_of_classes}"
    val = my_funcs.read_in_valid_filenames(fn_file.path)
    valid_filenames[key] = val



dataset: audioset, datasplit: eval, nr_of_classes: 30
dataset: audioset, datasplit: eval, nr_of_classes: 35
dataset: audioset, datasplit: eval, nr_of_classes: 40
dataset: audioset, datasplit: eval, nr_of_classes: 45
dataset: audioset, datasplit: eval, nr_of_classes: 50
dataset: audioset, datasplit: train, nr_of_classes: 30
dataset: audioset, datasplit: train, nr_of_classes: 35
dataset: audioset, datasplit: train, nr_of_classes: 40
dataset: audioset, datasplit: train, nr_of_classes: 45
dataset: audioset, datasplit: train, nr_of_classes: 50
dataset: fsd50k, datasplit: eval, nr_of_classes: 30
dataset: fsd50k, datasplit: eval, nr_of_classes: 35
dataset: fsd50k, datasplit: eval, nr_of_classes: 40
dataset: fsd50k, datasplit: eval, nr_of_classes: 45
dataset: fsd50k, datasplit: eval, nr_of_classes: 50
dataset: fsd50k, datasplit: train, nr_of_classes: 30
dataset: fsd50k, datasplit: train, nr_of_classes: 35
dataset: fsd50k, datasplit: train, nr_of_classes: 40
dataset: fsd50k, datasplit: train, n

In [10]:
label_files = os.scandir(PATH_TO_LABEL_DICT_DIR)

label_dicts = {}
for l_file in label_files:
    split_fname = l_file.name.split('_')
    dataset, datasplit, nr_of_classes = split_fname[0], split_fname[1], split_fname[3]
    print(f"dataset: {dataset}, datasplit: {datasplit}, nr_of_classes: {nr_of_classes}")
    key = f"{dataset}_{datasplit}_{nr_of_classes}"
    val = my_funcs.pickle_load(l_file.path)
    label_dicts[key] = val



dataset: audioset, datasplit: eval, nr_of_classes: 30
dataset: audioset, datasplit: eval, nr_of_classes: 35
dataset: audioset, datasplit: eval, nr_of_classes: 40
dataset: audioset, datasplit: eval, nr_of_classes: 45
dataset: audioset, datasplit: eval, nr_of_classes: 50
dataset: audioset, datasplit: train, nr_of_classes: 30
dataset: audioset, datasplit: train, nr_of_classes: 35
dataset: audioset, datasplit: train, nr_of_classes: 40
dataset: audioset, datasplit: train, nr_of_classes: 45
dataset: audioset, datasplit: train, nr_of_classes: 50
dataset: fsd50k, datasplit: eval, nr_of_classes: 30
dataset: fsd50k, datasplit: eval, nr_of_classes: 35
dataset: fsd50k, datasplit: eval, nr_of_classes: 40
dataset: fsd50k, datasplit: eval, nr_of_classes: 45
dataset: fsd50k, datasplit: eval, nr_of_classes: 50
dataset: fsd50k, datasplit: train, nr_of_classes: 30
dataset: fsd50k, datasplit: train, nr_of_classes: 35
dataset: fsd50k, datasplit: train, nr_of_classes: 40
dataset: fsd50k, datasplit: train, n

In [13]:
# Audioset has Some differences in filenames between audiofiles and metafiles
# FSD50k has varying length audiofiles which were processed into 10 sec chunks leading to filenames in the hdf5 being slightly different (an additional '_x', where x: {0, 1, 2}, suffix to identify it) when compared to valid filenames.

with h5py.File(PATH_TO_DATA_HDF5, 'a') as data_hdf5:

    for data_id in valid_filenames.keys():
        print(data_id)
        split_id = data_id.split('_')
        dataset, datasplit, nr_of_classes = split_id[0], split_id[1], split_id[2]
        print(f"dataset: {dataset}, datasplit: {datasplit}, nr_of_classes: {nr_of_classes}")

        filenames = valid_filenames[data_id]
        labels = label_dicts[data_id]

        grp_name = f"{dataset}_{datasplit}_50"
        grp = data_hdf5[grp_name]
        print(grp)

        for fname in grp.keys():

            fname_wo_Y = ""
            fname_og = ""
            if dataset == 'audioset':
                fname_wo_Y = fname[1::] # Remove leading Y
            else: # fsd50k
                if '_' in fname:
                    fname_og = fname.split('_')[0]
                else:
                    fname_og = fname

            membership_attr = f"in_{nr_of_classes}"
            label_attr = f"label_{nr_of_classes}"
            
            if dataset == 'audioset':
                if fname in filenames:
                    # Add two attributes: if the dataset is a part of lower classes
                    # and the matching label

                    grp[fname].attrs[membership_attr] = 1 # True
                    grp[fname].attrs[label_attr] = labels[fname_wo_Y]
                    
                else:

                    grp[fname].attrs[membership_attr] = 0 # False

            else: # fsd50k
                if fname_og in filenames:

                    grp[fname].attrs[membership_attr] = 1 # True
                    grp[fname].attrs[label_attr] = labels[fname_og]
                    
                else:
                    
                    grp[fname].attrs[membership_attr] = 0 # False


audioset_eval_30
dataset: audioset, datasplit: eval, nr_of_classes: 30
<HDF5 group "/audioset_eval_50" (14242 members)>
audioset_eval_35
dataset: audioset, datasplit: eval, nr_of_classes: 35
<HDF5 group "/audioset_eval_50" (14242 members)>
audioset_eval_40
dataset: audioset, datasplit: eval, nr_of_classes: 40
<HDF5 group "/audioset_eval_50" (14242 members)>
audioset_eval_45
dataset: audioset, datasplit: eval, nr_of_classes: 45
<HDF5 group "/audioset_eval_50" (14242 members)>
audioset_eval_50
dataset: audioset, datasplit: eval, nr_of_classes: 50
<HDF5 group "/audioset_eval_50" (14242 members)>
audioset_train_30
dataset: audioset, datasplit: train, nr_of_classes: 30
<HDF5 group "/audioset_train_50" (81070 members)>
audioset_train_35
dataset: audioset, datasplit: train, nr_of_classes: 35
<HDF5 group "/audioset_train_50" (81070 members)>
audioset_train_40
dataset: audioset, datasplit: train, nr_of_classes: 40
<HDF5 group "/audioset_train_50" (81070 members)>
audioset_train_45
dataset: audi

In [21]:
# For inspecting the hdf5 file
counter = 0
with h5py.File(PATH_TO_DATA_HDF5, 'r') as test:
    for grp in test.keys():
        tmp_grp = test[grp]
        print(tmp_grp)
        for file in list(tmp_grp.keys())[0:5]:
            tmp_ds = tmp_grp[file]
            if (tmp_ds.attrs['in_30'] == 1 and tmp_ds.attrs['in_30'] == 1):
                print(file)
                print(tmp_ds.attrs['label_30'])
                print(tmp_ds.attrs['label_35'])

                
            
    
    

<HDF5 group "/audioset_eval_50" (14242 members)>
Y--BfvyPmVMo
[1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Y--U7joUcTCo
[0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Y--i-y1v8Hy8
[1 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[1 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Y-0BIyqJj9ZU
[0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
<HDF5 group "/audioset_train_50" (81070 members)>
Y--24LG2mr-Y
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Y--5OkAjCI7g
[0 0 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Y--7UmfOkRbM
[1 0 1